# 505: Likelihood of No Further Warming After NZCO2 - Strict (Peak-Timing) Criterion

`501_nofurtherwarming_probability.ipynb` and `502_decline_statistics.ipynb` classify a scenario/ensemble
member as "no further warming after net-zero" whenever GSAT in 2100 is below GSAT in the year net-zero
CO2/GHG is reached (`delta_netzero_co2 < 0`). That criterion is **too permissive**: it also passes members
whose GSAT keeps rising for some years *after* net-zero (peak occurs after the net-zero year) and only
later declines back below the net-zero-year level by 2100. For those members there clearly *was* further
warming after net-zero, even though the 2100 endpoint looks favourable.

This notebook applies a stricter, peak-timing-based criterion and reports the likelihood of three
mutually exclusive, exhaustive member-level outcomes relative to the net-zero year:

1. **No further warming after NZCO2** (`peak_year <= year_netzero_co2` AND `GSAT|2100 <= GSAT|netzero`):
   both the peak does not occur after net-zero, and 2100 GSAT does not exceed the net-zero-year level.
2. **Peak and decline after NZCO2** (`peak_year > year_netzero_co2` and `GSAT|2100 < GSAT|netzero`):
   warming continues to rise after net-zero, but 2100 GSAT still ends up below the net-zero-year level.
   These are the members that the old, endpoint-based criterion in 501/502 *incorrectly* counted as
   "no further warming".
3. **No decline after NZCO2** (everything else): 2100 GSAT is at or above the net-zero-year level,
   whether or not the peak occurred after net-zero (this also captures the edge case of a member that
   peaks before net-zero but ticks up marginally above the net-zero-year level by 2100 without ever
   exceeding its earlier peak).

Likelihoods are computed per scenario (share of ensemble members satisfying each criterion) and then
summarised across scenarios by scenario group, following the same methodology as `501` (per-scenario
probability) and `503` (IPCC-calibrated-language style table: share of scenarios with >50%/>66%/>90% of
members satisfying a criterion).

**Postponed:** quantifying the *magnitude* of the overshoot (e.g. as cumulative "overshoot degree-years"
above the net-zero baseline) requires full annual GSAT trajectories per ensemble member, which are not
available locally for the full ~1,955-scenario production run (only per-member peak/net-zero-year/2100
point extracts were kept; the full per-member timeseries stayed on the cluster).

In [57]:
import os
from pathlib import Path

import dotenv
import pandas as pd
import matplotlib.pyplot as plt

dotenv.load_dotenv()

True

In [58]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
PLOT_FOLDER = OUTPUT_FOLDER / "plots"
PLOT_FOLDER.mkdir(parents=True, exist_ok=True)

# Input metrics file - the extended, group-labelled file produced by 501
MANIFEST_ID = "regenerated"
METRICS_FILE = OUTPUT_FOLDER / f"500b_component_metrics_extended_{MANIFEST_ID}.csv"

GROUPS = [
    "Scenarios (NZCO2)",
    "Scenarios (NZGHG)",
    "Stylised Variants (NZCO2-hold)",
    "Stylised Variants (NZGHG-hold)",
]

RENAME_MAP = {
    "Scenarios (NZCO2)":         "Scenarios\n(NZCO2)",
    "Scenarios (NZGHG)":         "Scenarios\n(NZGHG)",
    "Stylised Variants (NZCO2-hold)": "Stylised Variants\n(NZCO2-hold)",
    "Stylised Variants (NZGHG-hold)": "Stylised Variants\n(NZGHG-hold)",
}

## Load Data and Classify Ensemble Members

In [59]:
metrics_df = pd.read_csv(METRICS_FILE, index_col=0)
print(f"Total rows: {len(metrics_df)}")
metrics_df["scenario_group"].value_counts()

Total rows: 1405200


scenario_group
Scenarios (NZCO2)                 470400
Stylised Variants (NZCO2-hold)    470400
Stylised Variants (NZGHG-hold)    232200
Scenarios (NZGHG)                 232200
Name: count, dtype: int64

In [60]:
peak_before_nzco2 = metrics_df["peak_year"] <= metrics_df["year_netzero_co2"]
declines_by_2100 = metrics_df["GSAT|ANTHROPOGENIC|delta_netzero_co2"] < 0

# No further warming after NZCO2: BOTH peak GSAT at or before the net-zero year AND 2100 GSAT
# at or below the net-zero-year level (a member that peaks early but still ticks up marginally
# above the net-zero-year level by 2100 does NOT qualify - it falls into "no decline after NZCO2" below)
metrics_df["no_further_warming_after_nzco2"] = peak_before_nzco2 & (metrics_df["GSAT|ANTHROPOGENIC|delta_netzero_co2"] <= 0)

# Peak and decline after NZCO2: peak occurs after net-zero, but 2100 still ends up below the net-zero-year level
metrics_df["peak_and_decline_after_nzco2"] = (~peak_before_nzco2) & declines_by_2100

# No decline after NZCO2: everything else - 2100 GSAT is at or above the net-zero-year level,
# whether or not the peak occurred after net-zero
metrics_df["no_decline_after_nzco2"] = ~(
    metrics_df["no_further_warming_after_nzco2"] | metrics_df["peak_and_decline_after_nzco2"]
)

# Sanity check: the three categories are mutually exclusive and exhaustive
category_sum = (
    metrics_df["no_further_warming_after_nzco2"].astype(int)
    + metrics_df["peak_and_decline_after_nzco2"].astype(int)
    + metrics_df["no_decline_after_nzco2"].astype(int)
)
assert (category_sum == 1).all(), "Categories are not mutually exclusive/exhaustive"

print("Member-level classification (all groups combined):")
metrics_df[["no_further_warming_after_nzco2", "peak_and_decline_after_nzco2", "no_decline_after_nzco2"]].mean().mul(100).round(1)


Member-level classification (all groups combined):


no_further_warming_after_nzco2    65.9
peak_and_decline_after_nzco2      15.6
no_decline_after_nzco2            18.5
dtype: float64

## Per-Scenario Probability and Group Summary Helpers

Following `501`'s methodology: for each scenario, compute the share of ensemble members satisfying a
given criterion, then summarise that per-scenario probability distribution across scenarios within each
scenario group using percentile ranges (median, p34-p67, p17-p83, p10-p90, p05-p95) - these ranges map
directly onto IPCC calibrated likelihood language (e.g. p17-p83 = "likely", p05-p95 = "very likely").


In [61]:
def per_scenario_probability(df, criterion_col):
    """Per-scenario share of ensemble members satisfying criterion_col."""
    prob = df.groupby(["scenario_group", "model", "scenario"]).agg(
        n_ensemble=pd.NamedAgg(column=criterion_col, aggfunc="count"),
        n_true=pd.NamedAgg(column=criterion_col, aggfunc="sum"),
    ).reset_index()
    prob["prob_pct"] = (prob["n_true"] / prob["n_ensemble"] * 100).round(1)
    return prob


def build_probability_range_table(prob_df, groups=GROUPS):
    """501-style percentile-range summary of a per-scenario probability column, across scenarios."""
    def format_row_values(group_data):
        return [
            f"{group_data.median():.1f}%",
            f"({group_data.quantile(0.34):.1f}% to {group_data.quantile(0.67):.1f}%)",
            f"({group_data.quantile(0.17):.1f}% to {group_data.quantile(0.83):.1f}%)",
            f"({group_data.quantile(0.10):.1f}% to {group_data.quantile(0.90):.1f}%)",
            f"({group_data.quantile(0.05):.1f}% to {group_data.quantile(0.95):.1f}%)",
            f"{len(group_data)}",
        ]

    stat_labels = [
        "Median [%]",
        "(p34 to p67) [%]",
        "(p17 to p83) [%]",
        "(p10 to p90) [%]",
        "(p05 to p95) [%]",
        "N scenarios",
    ]

    rows = []
    for stat_label, val_idx in zip(stat_labels, range(len(stat_labels))):
        row = {"Statistic": stat_label}
        for grp in groups:
            group_data = prob_df[prob_df["scenario_group"] == grp]["prob_pct"]
            row[grp] = format_row_values(group_data)[val_idx] if len(group_data) else "—"
        rows.append(row)

    display_table = pd.DataFrame(rows).set_index("Statistic")
    display_table.columns = display_table.columns.map(RENAME_MAP)
    return display_table


def save_likelihood_table_image(display_table, path, title=None):
    n_rows = len(display_table)
    n_cols = len(display_table.columns)
    fig, ax = plt.subplots(figsize=(6, n_rows * 0.5))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=11)
    table = ax.table(
        cellText=display_table.values,
        colLabels=display_table.columns,
        rowLabels=list(display_table.index),
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    header_height = 0.15
    for col_idx in range(n_cols):
        table[0, col_idx].set_height(header_height)
        table[0, col_idx].get_text().set_fontweight("bold")
        table[0, col_idx].set_facecolor("#e0e0e0")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

## Likelihood Table: No Further Warming After NZCO2 (Strict Criterion)

In [62]:
strict_prob = per_scenario_probability(metrics_df, "no_further_warming_after_nzco2")
strict_likelihood_display = build_probability_range_table(strict_prob)

print("Likelihood of no further warming after NZCO2")
strict_likelihood_display

Likelihood of no further warming after NZCO2


,Scenarios\n(NZCO2),Scenarios\n(NZGHG),Stylised Variants\n(NZCO2-hold),Stylised Variants\n(NZGHG-hold)
Statistic,,,,
Median [%],70.2%,74.0%,68.4%,73.3%
(p34 to p67) [%],(59.2% to 76.9%),(66.3% to 78.5%),(56.2% to 75.8%),(66.3% to 78.3%)
(p17 to p83) [%],(45.8% to 85.0%),(49.0% to 85.1%),(44.3% to 82.7%),(48.6% to 85.0%)
(p10 to p90) [%],(36.9% to 88.4%),(45.7% to 90.1%),(35.3% to 87.2%),(45.5% to 90.1%)
(p05 to p95) [%],(22.4% to 93.7%),(37.8% to 95.1%),(22.0% to 92.7%),(37.8% to 94.5%)
N scenarios,784,387,784,387


In [63]:
save_likelihood_table_image(
    strict_likelihood_display,
    PLOT_FOLDER / "505_strict_nfw_likelihood_table.png",
    title="Likelihood of no further warming after NZCO2",
)

## Likelihood Table: Peak and Decline After NZCO2

This is the group that 501/502's endpoint-only criterion incorrectly counted as "no further warming": GSAT
keeps rising after net-zero (peak occurs after the net-zero year), but still ends up below the net-zero-year
level by 2100.

In [64]:
gap_prob = per_scenario_probability(metrics_df, "peak_and_decline_after_nzco2")
gap_likelihood_display = build_probability_range_table(gap_prob)

print("Likelihood of post-NZCO2 peak and decline below NZCO2 GSAT by 2100")
gap_likelihood_display

Likelihood of post-NZCO2 peak and decline below NZCO2 GSAT by 2100


,Scenarios\n(NZCO2),Scenarios\n(NZGHG),Stylised Variants\n(NZCO2-hold),Stylised Variants\n(NZGHG-hold)
Statistic,,,,
Median [%],13.1%,20.5%,3.5%,17.8%
(p34 to p67) [%],(7.0% to 20.6%),(16.5% to 27.9%),(1.3% to 6.8%),(14.0% to 23.8%)
(p17 to p83) [%],(0.7% to 32.8%),(10.3% to 38.0%),(0.2% to 12.8%),(8.9% to 33.1%)
(p10 to p90) [%],(0.0% to 40.2%),(6.9% to 46.3%),(0.0% to 20.6%),(5.7% to 38.5%)
(p05 to p95) [%],(0.0% to 51.7%),(2.7% to 52.3%),(0.0% to 28.4%),(2.0% to 50.3%)
N scenarios,784,387,784,387


In [65]:
save_likelihood_table_image(
    gap_likelihood_display,
    PLOT_FOLDER / "505_overshoot_then_decline_likelihood_table.png",
    title="Likelihood of post-NZCO2 peak and decline\nbelow NZCO2 GSAT by 2100",
)

## Sensitivity Check: Population = Every Ensemble Member of Every Scenario

The two tables above summarise a per-scenario probability (share of that scenario's members
satisfying a criterion), then take percentiles of that per-scenario probability *across scenarios*.
As a sensitivity check: what happens if we instead pool every (scenario, member) row within a
scenario group into one flat population, with no per-scenario grouping step at all, and compute
the median/percentile range directly on that pooled population?

Each pooled observation is a raw 0/1 (criterion satisfied or not). A percentile of a 0/1 population
can only ever be exactly 0% or 100% (there is no partial value an individual member can take), so
**every percentile row below is degenerate by construction** - it just reports whether the pooled
share of `True` values is above or below that percentile threshold. The only informative statistic
from a flat pooled population is the **mean** (= overall pooled proportion satisfying the criterion),
shown for comparison.

In [66]:
def build_pooled_population_table(df, criterion_col, groups=GROUPS):
    """Pool every (scenario, member) row per scenario_group into one flat population and
    report the mean plus percentiles directly on that pooled 0/1 population (percentiles are
    degenerate - included only to make that degeneracy explicit)."""
    stat_labels = [
        "Mean (pooled) [%]",
        "Median (pooled) [%]",
        "(p17 to p83) (pooled) [%]",
        "(p05 to p95) (pooled) [%]",
        "N members (pooled)",
    ]
    rows = []
    for stat_label in stat_labels:
        row = {"Statistic": stat_label}
        for grp in groups:
            pop = df[df["scenario_group"] == grp][criterion_col].astype(float) * 100
            if stat_label == "Mean (pooled) [%]":
                row[grp] = f"{pop.mean():.1f}%"
            elif stat_label == "Median (pooled) [%]":
                row[grp] = f"{pop.median():.1f}%"
            elif stat_label == "(p17 to p83) (pooled) [%]":
                row[grp] = f"({pop.quantile(0.17):.1f}% to {pop.quantile(0.83):.1f}%)"
            elif stat_label == "(p05 to p95) (pooled) [%]":
                row[grp] = f"({pop.quantile(0.05):.1f}% to {pop.quantile(0.95):.1f}%)"
            elif stat_label == "N members (pooled)":
                row[grp] = f"{len(pop)}"
        rows.append(row)
    display_table = pd.DataFrame(rows).set_index("Statistic")
    display_table.columns = display_table.columns.map(RENAME_MAP)
    return display_table

In [67]:
strict_pooled_display = build_pooled_population_table(metrics_df, "no_further_warming_after_nzco2")

print("Pooled-population sensitivity check: no further warming after NZCO2")
strict_pooled_display

Pooled-population sensitivity check: no further warming after NZCO2


,Scenarios\n(NZCO2),Scenarios\n(NZGHG),Stylised Variants\n(NZCO2-hold),Stylised Variants\n(NZGHG-hold)
Statistic,,,,
Mean (pooled) [%],65.2%,69.1%,63.5%,69.0%
Median (pooled) [%],100.0%,100.0%,100.0%,100.0%
(p17 to p83) (pooled) [%],(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%)
(p05 to p95) (pooled) [%],(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%)
N members (pooled),470400,232200,470400,232200


In [68]:
gap_pooled_display = build_pooled_population_table(metrics_df, "peak_and_decline_after_nzco2")

print("Pooled-population sensitivity check: peak and decline after NZCO2")
gap_pooled_display

Pooled-population sensitivity check: peak and decline after NZCO2


,Scenarios\n(NZCO2),Scenarios\n(NZGHG),Stylised Variants\n(NZCO2-hold),Stylised Variants\n(NZGHG-hold)
Statistic,,,,
Mean (pooled) [%],17.3%,24.0%,7.2%,21.0%
Median (pooled) [%],0.0%,0.0%,0.0%,0.0%
(p17 to p83) (pooled) [%],(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 0.0%),(0.0% to 100.0%)
(p05 to p95) (pooled) [%],(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%),(0.0% to 100.0%)
N members (pooled),470400,232200,470400,232200


In [69]:
save_likelihood_table_image(
    strict_pooled_display,
    PLOT_FOLDER / "505_strict_nfw_pooled_population_table.png",
    title="Sensitivity check: no further warming after NZCO2, pooled population (percentiles degenerate)",
)
save_likelihood_table_image(
    gap_pooled_display,
    PLOT_FOLDER / "505_overshoot_then_decline_pooled_population_table.png",
    title="Sensitivity check: peak and decline after NZCO2, pooled population (percentiles degenerate)",
)

## Sensitivity Check: Leave-One-Model-Out (No Further Warming After NZCO2)

The scenario database is far from a set of independent draws: for `Scenarios (NZCO2)`, 784
scenarios come from only 54 models, and one model alone (MESSAGEix-GLOBIOM 1.1) contributes 122 of
them (~16%); the median model contributes only 6. All 41 models behind the NZGHG-anchored groups
are a subset of the 54 NZCO2-anchored models, so a single model list covers all four groups.

As a cheap, fully deterministic alternative to a full cluster bootstrap: recompute the group median
(from the same per-scenario probabilities already computed above) after dropping one model's
scenarios at a time, for every model. This directly shows which model's presence moves the reported
median the most - a table, not a single interval, but every row is exact (no resampling) and each
row documents which model was excluded and the resulting N. Models are ordered by their scenario
count in `Scenarios (NZCO2)` (descending), so the models most likely to matter appear first. A model
not present in a given group leaves that group's column unchanged (same median, same N).

In [70]:
# All models across the full universe (NZGHG's 41 models are a strict subset of NZCO2's 54)
model_scenario_counts = (
    metrics_df[metrics_df["scenario_group"] == "Scenarios (NZCO2)"]
    [["model", "scenario"]].drop_duplicates()
    .groupby("model").size().sort_values(ascending=False)
)
ordered_models = model_scenario_counts.index.tolist()


def median_and_n(prob_df, group_name, exclude_model=None):
    data = prob_df[prob_df["scenario_group"] == group_name]
    if exclude_model is not None:
        data = data[data["model"] != exclude_model]
    if len(data) == 0:
        return None, 0
    return data["prob_pct"].median(), len(data)


rows = []
baseline_row = {"Model excluded": "(none - all 54 models)"}
for grp in GROUPS:
    med, n = median_and_n(strict_prob, grp)
    baseline_row[grp] = f"{med:.1f}% (n={n})"
rows.append(baseline_row)

for model in ordered_models:
    row = {"Model excluded": f"{model} (n_scen={model_scenario_counts[model]})"}
    for grp in GROUPS:
        med, n = median_and_n(strict_prob, grp, exclude_model=model)
        row[grp] = f"{med:.1f}% (n={n})" if med is not None else "—"
    rows.append(row)

loo_table = pd.DataFrame(rows).set_index("Model excluded")
loo_table.columns = loo_table.columns.map(RENAME_MAP)

print(f"Leave-one-model-out table: {len(loo_table)} rows (1 baseline + {len(ordered_models)} models)")
loo_table

Leave-one-model-out table: 55 rows (1 baseline + 54 models)


,Scenarios\n(NZCO2),Scenarios\n(NZGHG),Stylised Variants\n(NZCO2-hold),Stylised Variants\n(NZGHG-hold)
Model excluded,,,,
(none - all 54 models),70.2% (n=784),74.0% (n=387),68.4% (n=784),73.3% (n=387)
MESSAGEix-GLOBIOM 1.1 (n_scen=122),67.7% (n=662),72.9% (n=316),64.3% (n=662),72.7% (n=316)
WITCH 5.0 (n_scen=56),71.5% (n=728),74.7% (n=362),69.5% (n=728),74.6% (n=362)
REMIND-MAgPIE 2.1-4.2 (n_scen=50),68.8% (n=734),73.0% (n=370),66.8% (n=734),72.9% (n=370)
POLES-JRC ENGAGE (n_scen=39),69.5% (n=745),72.7% (n=363),67.7% (n=745),72.3% (n=363)
AIM/CGE V2.2 (n_scen=36),70.2% (n=748),74.0% (n=377),68.7% (n=748),73.5% (n=377)
COFFEE 1.1 (n_scen=35),70.2% (n=749),74.0% (n=355),68.5% (n=749),74.0% (n=355)
GCAM 5.3 (n_scen=35),71.3% (n=749),74.0% (n=386),69.3% (n=749),73.4% (n=386)
IMAGE 3.2 (n_scen=34),70.3% (n=750),74.0% (n=371),69.0% (n=750),73.5% (n=371)


In [78]:
def save_leave_one_model_out_table_image(display_table, path, title=None):
    n_rows = len(display_table)
    n_cols = len(display_table.columns)
    fig, ax = plt.subplots(figsize=(8, n_rows * 0.22 + 0.7))
    ax.axis("off")
    table = ax.table(
        cellText=display_table.values,
        colLabels=display_table.columns,
        rowLabels=list(display_table.index),
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.0)
    header_height = 0.04
    for col_idx in range(n_cols):
        table[0, col_idx].set_height(header_height)
        table[0, col_idx].get_text().set_fontweight("bold")
        table[0, col_idx].set_facecolor("#e0e0e0")
    # Highlight the baseline row
    for col_idx in range(n_cols):
        table[1, col_idx].set_facecolor("#f0f0f0")
    if title:
        fig.suptitle(title, fontsize=12, y=0.9)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


save_leave_one_model_out_table_image(
    loo_table,
    PLOT_FOLDER / "505_strict_nfw_leave_one_model_out_table.png",
    title="Sensitivity check: no further warming after NZCO2,\nmedian with one model excluded at a time",
)